In [1]:
! pip install -q GraKeL

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 8.5 MB/s eta 0:00:00


In [2]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.manifold import Isomap, SpectralEmbedding, LocallyLinearEmbedding

from grakel.datasets import fetch_dataset
from grakel.kernels import ShortestPath, GraphletSampling, RandomWalk, WeisfeilerLehman
from grakel.datasets.base import read_data
import gdown
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# https://drive.google.com/file/d/1o-v6-1AVQAsVrJNQ61QEdYMHS0T2gegN/view?usp=sharing
# https://drive.google.com/file/d/1iThfJIEDQOjh3atQCsCqFwjOAwoQ0N4C/view?usp=sharing
# https://drive.google.com/file/d/1CZsYHeDOlxcRjf7naZ9w_9_f4PTDjUWc/view?usp=sharing

In [3]:
gdown.download('http://drive.google.com/uc?export=view&id=1CZsYHeDOlxcRjf7naZ9w_9_f4PTDjUWc', './mutag.zip', quiet = True)
gdown.download('http://drive.google.com/uc?export=view&id=1iThfJIEDQOjh3atQCsCqFwjOAwoQ0N4C', './PTC_MR.zip', quiet = True)
gdown.download('http://drive.google.com/uc?export=view&id=1o-v6-1AVQAsVrJNQ61QEdYMHS0T2gegN', './PROTEINS.zip', quiet = True)

'./PROTEINS.zip'

In [4]:
!unzip -q ./mutag.zip
!unzip -q ./PTC_MR.zip
!unzip -q ./PROTEINS.zip

# MUTAG Dataset

In [ ]:
MUTAG = read_data("MUTAG")
graph_data, y = np.array(MUTAG.data), np.array(MUTAG.target)
G_train, G_test, y_train, y_test = train_test_split(graph_data, y, test_size=0.1, random_state=42)

## with Manifold Learning

In [ ]:
kernels = [ShortestPath, GraphletSampling, RandomWalk, WeisfeilerLehman]
manifolds = [Isomap, LocallyLinearEmbedding, SpectralEmbedding]
kernels_names = ['ShortestPath', 'GraphletSampling', 'RandomWalk', 'WeisfeilerLehman']
manifolds_names = ['Isomap', 'LocallyLinearEmbedding', 'SpectralEmbedding']
for i, kernel in enumerate(kernels):
    g_kernel = kernel(normalize = True)
    K_train = g_kernel.fit_transform(G_train)
    K_test = g_kernel.transform(G_test)
    for j, manif in enumerate(manifolds):
        encoder = manif(n_components = 10)
        E_train = encoder.fit_transform(K_train)
        E_test = encoder.transform(K_test) if j < 2 else encoder.fit_transform(K_test)

        model = SVC()
        model.fit(E_train, y_train)
        y_pred = model.predict(E_test)

        # Computes and prints the classification accuracy
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        print(f"{kernels_names[i]} + {manifolds_names[j]} ===> acuuracy: {round(acc, 4)}, precision: {round(prec, 4)}, recall: {round(rec, 4)}")

ShortestPath + Isomap ===> acuuracy: 0.9474, precision: 0.9231, recall: 1.0
ShortestPath + LocallyLinearEmbedding ===> acuuracy: 0.9474, precision: 0.9231, recall: 1.0
ShortestPath + SpectralEmbedding ===> acuuracy: 0.6316, precision: 0.6316, recall: 1.0
GraphletSampling + Isomap ===> acuuracy: 0.7368, precision: 0.7059, recall: 1.0
GraphletSampling + LocallyLinearEmbedding ===> acuuracy: 0.7368, precision: 0.7059, recall: 1.0
GraphletSampling + SpectralEmbedding ===> acuuracy: 0.6316, precision: 0.6316, recall: 1.0
RandomWalk + Isomap ===> acuuracy: 1.0, precision: 1.0, recall: 1.0
RandomWalk + LocallyLinearEmbedding ===> acuuracy: 1.0, precision: 1.0, recall: 1.0
RandomWalk + SpectralEmbedding ===> acuuracy: 0.6316, precision: 0.6316, recall: 1.0
WeisfeilerLehman + Isomap ===> acuuracy: 0.8947, precision: 0.9167, recall: 0.9167
WeisfeilerLehman + LocallyLinearEmbedding ===> acuuracy: 0.9474, precision: 1.0, recall: 0.9167
WeisfeilerLehman + SpectralEmbedding ===> acuuracy: 0.6316, pr

## without Manifold Learning

In [ ]:
kernels = [ShortestPath, GraphletSampling, RandomWalk, WeisfeilerLehman]
manifolds = [Isomap, LocallyLinearEmbedding, SpectralEmbedding]
kernels_names = ['ShortestPath', 'GraphletSampling', 'RandomWalk', 'WeisfeilerLehman']
manifolds_names = ['Isomap', 'LocallyLinearEmbedding', 'SpectralEmbedding']
for i, kernel in enumerate(kernels):
    g_kernel = kernel(normalize = True)
    K_train = g_kernel.fit_transform(G_train)
    K_test = g_kernel.transform(G_test)

    model = SVC()
    model.fit(K_train, y_train)
    y_pred = model.predict(K_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    print(f"{kernels_names[i]} ===> acuuracy: {round(acc, 4)}, precision: {round(prec, 4)}, recall: {round(rec, 4)}")

ShortestPath ===> acuuracy: 0.9474, precision: 0.9231, recall: 1.0
GraphletSampling ===> acuuracy: 0.6842, precision: 0.6667, recall: 1.0
RandomWalk ===> acuuracy: 1.0, precision: 1.0, recall: 1.0
WeisfeilerLehman ===> acuuracy: 0.8947, precision: 0.9167, recall: 0.9167


# PTC Dataset

In [ ]:
PTC = read_data("PTC_MR")
graph_data, y = PTC.data, PTC.target
G_train, G_test, y_train, y_test = train_test_split(graph_data, y, test_size=0.1, random_state=42)

## with Manifold Learning

In [ ]:
kernels = [ShortestPath, GraphletSampling, RandomWalk, WeisfeilerLehman]
manifolds = [Isomap, LocallyLinearEmbedding, SpectralEmbedding]
kernels_names = ['ShortestPath', 'GraphletSampling', 'RandomWalk', 'WeisfeilerLehman']
manifolds_names = ['Isomap', 'LocallyLinearEmbedding', 'SpectralEmbedding']
for i, kernel in enumerate(kernels):
    g_kernel = kernel(normalize = True if not i in [1,2] else False)
    K_train = g_kernel.fit_transform(G_train)
    K_test = g_kernel.transform(G_test)
    for j, manif in enumerate(manifolds):
        encoder = manif(n_components = 10)
        E_train = encoder.fit_transform(K_train)
        E_test = encoder.transform(K_test) if j < 2 else encoder.fit_transform(K_test)

        model = SVC()
        model.fit(E_train, y_train)
        y_pred = model.predict(E_test)

        # Computes and prints the classification accuracy
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        print(f"{kernels_names[i]} + {manifolds_names[j]} ===> acuuracy: {round(acc, 4)}, precision: {round(prec, 4)}, recall: {round(rec, 4)}")

ShortestPath + Isomap ===> acuuracy: 0.5143, precision: 0.3333, recall: 0.1333
ShortestPath + LocallyLinearEmbedding ===> acuuracy: 0.5714, precision: 0.5, recall: 0.3333
ShortestPath + SpectralEmbedding ===> acuuracy: 0.5714, precision: 0.0, recall: 0.0
GraphletSampling + Isomap ===> acuuracy: 0.7143, precision: 0.8571, recall: 0.4
GraphletSampling + LocallyLinearEmbedding ===> acuuracy: 0.5714, precision: 0.5, recall: 0.1333
GraphletSampling + SpectralEmbedding ===> acuuracy: 0.5714, precision: 0.0, recall: 0.0
RandomWalk + Isomap ===> acuuracy: 0.6571, precision: 0.7143, recall: 0.3333
RandomWalk + LocallyLinearEmbedding ===> acuuracy: 0.4571, precision: 0.0, recall: 0.0
RandomWalk + SpectralEmbedding ===> acuuracy: 0.5714, precision: 0.0, recall: 0.0
WeisfeilerLehman + Isomap ===> acuuracy: 0.6286, precision: 0.75, recall: 0.2
WeisfeilerLehman + LocallyLinearEmbedding ===> acuuracy: 0.6286, precision: 0.6667, recall: 0.2667
WeisfeilerLehman + SpectralEmbedding ===> acuuracy: 0.5714

## without Manifold Learning

In [ ]:
kernels = [ShortestPath, GraphletSampling, RandomWalk, WeisfeilerLehman]
manifolds = [Isomap, LocallyLinearEmbedding]
kernels_names = ['ShortestPath', 'GraphletSampling', 'RandomWalk', 'WeisfeilerLehman']
manifolds_names = ['Isomap', 'LocallyLinearEmbedding']
for i, kernel in enumerate(kernels):
    g_kernel = kernel(normalize = True if not i in [1,2] else False)
    K_train = g_kernel.fit_transform(G_train)
    K_test = g_kernel.transform(G_test)

    model = SVC()
    model.fit(K_train, y_train)
    y_pred = model.predict(K_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    print(f"{kernels_names[i]} ===> acuuracy: {round(acc, 4)}, precision: {round(prec, 4)}, recall: {round(rec, 4)}")

ShortestPath ===> acuuracy: 0.6286, precision: 0.75, recall: 0.2
GraphletSampling ===> acuuracy: 0.7429, precision: 0.875, recall: 0.4667
RandomWalk ===> acuuracy: 0.6, precision: 0.6, recall: 0.2
WeisfeilerLehman ===> acuuracy: 0.6571, precision: 0.8, recall: 0.2667


# PROTEINS Dataset

In [ ]:
PROTEINS = read_data("PROTEINS")
graph_data, y = PTC.data, PTC.target
G_train, G_test, y_train, y_test = train_test_split(graph_data, y, test_size=0.1, random_state=42)

## with Manifold Learning

In [ ]:
kernels = [ShortestPath, GraphletSampling, RandomWalk, WeisfeilerLehman]
manifolds = [Isomap, LocallyLinearEmbedding, SpectralEmbedding]
kernels_names = ['ShortestPath', 'GraphletSampling', 'RandomWalk', 'WeisfeilerLehman']
manifolds_names = ['Isomap', 'LocallyLinearEmbedding', 'SpectralEmbedding']
for i, kernel in enumerate(kernels):
    g_kernel = kernel(normalize = True if not i in [1,2] else False)
    K_train = g_kernel.fit_transform(G_train)
    K_test = g_kernel.transform(G_test)
    for j, manif in enumerate(manifolds):
        encoder = manif(n_components = 10)
        E_train = encoder.fit_transform(K_train)
        E_test = encoder.transform(K_test) if j < 2 else encoder.fit_transform(K_test)

        model = SVC()
        model.fit(E_train, y_train)
        y_pred = model.predict(E_test)

        # Computes and prints the classification accuracy
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        print(f"{kernels_names[i]} + {manifolds_names[j]} ===> acuuracy: {round(acc, 4)}, precision: {round(prec, 4)}, recall: {round(rec, 4)}")

ShortestPath + Isomap ===> acuuracy: 0.5143, precision: 0.3333, recall: 0.1333
ShortestPath + LocallyLinearEmbedding ===> acuuracy: 0.5714, precision: 0.5, recall: 0.3333
ShortestPath + SpectralEmbedding ===> acuuracy: 0.5714, precision: 0.0, recall: 0.0
GraphletSampling + Isomap ===> acuuracy: 0.7143, precision: 0.8571, recall: 0.4
GraphletSampling + LocallyLinearEmbedding ===> acuuracy: 0.5714, precision: 0.5, recall: 0.1333
GraphletSampling + SpectralEmbedding ===> acuuracy: 0.5714, precision: 0.0, recall: 0.0
RandomWalk + Isomap ===> acuuracy: 0.6571, precision: 0.7143, recall: 0.3333
RandomWalk + LocallyLinearEmbedding ===> acuuracy: 0.4571, precision: 0.0, recall: 0.0
RandomWalk + SpectralEmbedding ===> acuuracy: 0.5714, precision: 0.0, recall: 0.0
WeisfeilerLehman + Isomap ===> acuuracy: 0.6286, precision: 0.75, recall: 0.2
WeisfeilerLehman + LocallyLinearEmbedding ===> acuuracy: 0.6286, precision: 0.6667, recall: 0.2667
WeisfeilerLehman + SpectralEmbedding ===> acuuracy: 0.5714

## without Manifold Learning

In [ ]:
kernels = [ShortestPath, GraphletSampling, RandomWalk, WeisfeilerLehman]
manifolds = [Isomap, LocallyLinearEmbedding]
kernels_names = ['ShortestPath', 'GraphletSampling', 'RandomWalk', 'WeisfeilerLehman']
manifolds_names = ['Isomap', 'LocallyLinearEmbedding']
for i, kernel in enumerate(kernels):
    g_kernel = kernel(normalize = True if not i in [1,2] else False)
    K_train = g_kernel.fit_transform(G_train)
    K_test = g_kernel.transform(G_test)

    model = SVC()
    model.fit(K_train, y_train)
    y_pred = model.predict(K_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    print(f"{kernels_names[i]} ===> acuuracy: {round(acc, 4)}, precision: {round(prec, 4)}, recall: {round(rec, 4)}")

ShortestPath ===> acuuracy: 0.6286, precision: 0.75, recall: 0.2
GraphletSampling ===> acuuracy: 0.7429, precision: 0.875, recall: 0.4667
RandomWalk ===> acuuracy: 0.6, precision: 0.6, recall: 0.2
WeisfeilerLehman ===> acuuracy: 0.6571, precision: 0.8, recall: 0.2667
